In [12]:
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
import numpy as np
class modified_linear(nn.Linear):
    def __init__(self, in_features, out_features,mode = 'normal', bias = True, device=None, dtype=None):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long)  
        )
        self.hashtable = None 
    
    def prune(self,threshold):
        self.mode = 'prune'
        new_mask = (self.weight.abs() > threshold).float()   
        self.mask.mul_(new_mask)
        print("pruned_linear!!")
        return
    
    def quantize(self,k):
        self.mode = 'quantize'
        print("quantized_linear!!")
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )
        kmeans.fit(matrix.reshape(-1,1))
        assignments = (
            torch.from_numpy(kmeans.labels_)
            .reshape(self.weight.shape)
            .to(torch.uint8)
        )
        
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        with torch.no_grad():
            self.weight.data = torch.empty(0, device=self.weight.device)
        return
    
    def forward(self, input):
        if(self.mode == 'normal'):
            W = self.weight 
        elif(self.mode == 'prune'):
            W = (self.weight * self.mask)
        elif(self.mode == 'quantize'):
            W = self.hashtable[self.assignments] * self.mask 
        return F.linear(input,W,self.bias)


In [13]:
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

class modified_conv2d(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=True,
        padding_mode="zeros",
        mode="normal",
        device=None,
        dtype=None,
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            padding_mode,
            device=device,
            dtype=dtype,
        )
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long),
        )
        self.hashtable = None 

    def prune(self, threshold):
        self.mode = "prune"
        new_mask = (self.weight.abs() > threshold).float()
        self.mask.mul_(new_mask)  
        print("pruned_conv2d!!")
        return

    def quantize(self, k):
        self.mode = "quantize"
        print("quantized_conv2d")
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,
        )
        kmeans.fit(matrix.reshape(-1, 1))
        # assignments = torch.from_numpy(
        #     kmeans.labels_
        # ).reshape(self.weight.shape).long() #original code
        assignments = (
            torch.from_numpy(kmeans.labels_)
            .reshape(self.weight.shape)
            .to(torch.uint8)
        )
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        with torch.no_grad():
            self.weight.data = torch.empty(0, device=self.weight.device)
        return

    def forward(self, input):
        if self.mode == "normal":
            W = self.weight
        elif self.mode == "prune":
            W = self.weight * self.mask
        elif self.mode == "quantize":
            W = self.hashtable[self.assignments] * self.mask
        return F.conv2d(
            input,
            W,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )

In [14]:
import torch
from torch import nn

def prune_model(model, prune_fraction):
    assert 0.0 < prune_fraction < 1.0
    all_weights = []
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.detach().abs().flatten()
            all_weights.append(w)
    all_weights = torch.cat(all_weights)

    k = int(prune_fraction * all_weights.numel())
    threshold = torch.kthvalue(all_weights, k).values.item()
    model.prune(threshold)
    return 

def quantize_model(model,K):
    model.quantize(K)
    return
def quantize_model(model,K):
    model.quantize(K)
    return

In [15]:
import cv2
import os
def compute_shape_features(binary_mask):
    contours, _ = cv2.findContours(
        binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return np.zeros(6, dtype=np.float32)

    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)

    if area == 0:
        return np.zeros(6, dtype=np.float32)

    # Area ratio
    h, w = binary_mask.shape
    area_ratio = area / (h * w)

    # Aspect ratio
    x, y, bw, bh = cv2.boundingRect(cnt)
    aspect_ratio = bw / bh if bh != 0 else 0

    # Solidity
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area != 0 else 0

    # Circularity
    perimeter = cv2.arcLength(cnt, True)
    circularity = (4 * np.pi * area) / (perimeter ** 2 + 1e-6)

    # Hu moments
    moments = cv2.moments(cnt)
    hu = cv2.HuMoments(moments).flatten()[:2]

    return np.array([
        area_ratio,
        aspect_ratio,
        solidity,
        circularity,
        hu[0],
        hu[1]
    ], dtype=np.float32)

def compute_lbp(gray):
    center = gray[1:-1, 1:-1]
    lbp = np.zeros_like(center, dtype=np.uint8)

    lbp |= (gray[:-2, :-2] >= center) << 7
    lbp |= (gray[:-2, 1:-1] >= center) << 6
    lbp |= (gray[:-2, 2:] >= center) << 5
    lbp |= (gray[1:-1, 2:] >= center) << 4
    lbp |= (gray[2:, 2:] >= center) << 3
    lbp |= (gray[2:, 1:-1] >= center) << 2
    lbp |= (gray[2:, :-2] >= center) << 1
    lbp |= (gray[1:-1, :-2] >= center) << 0

    return lbp

In [17]:
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

class Fruits360Dataset(Dataset):
    def __init__(self, root_dir, image_size=100):
        self.samples = []
        self.image_size = image_size

        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(sorted(os.listdir(root_dir)))
        }

        for cls in self.class_to_idx:
            cls_path = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_path):
                self.samples.append((
                    os.path.join(cls_path, img_name),
                    self.class_to_idx[cls]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        # ---- Read RGB ----
        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.image_size, self.image_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0

        # ---- Grayscale ----
        gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

        # ---- LBP ----
        lbp = compute_lbp(gray)
        lbp = cv2.resize(lbp, (self.image_size, self.image_size))
        lbp = lbp.astype(np.float32) / 255.0

        # ---- Edge ----
        edges = cv2.Canny(gray, 80, 160)
        edges = edges.astype(np.float32) / 255.0

        # ---- Color Features ----
        hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
        hsv_mean = hsv.mean(axis=(0, 1))
        hsv_std = hsv.std(axis=(0, 1))
        color_features = np.concatenate([hsv_mean, hsv_std])

        # ---- Shape Features ----
        _, binary = cv2.threshold(gray, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        shape_features = compute_shape_features(binary)

        # ---- Convert to tensors ----
        rgb_tensor = torch.from_numpy(img).permute(2, 0, 1)
        lbp_tensor = torch.from_numpy(lbp).unsqueeze(0)
        edge_tensor = torch.from_numpy(edges).unsqueeze(0)

        linear_features = torch.from_numpy(
            np.concatenate([color_features, shape_features]).astype(np.float32)
        )

        return rgb_tensor, lbp_tensor, edge_tensor, linear_features, label

In [ ]:
import torch
from torch import nn
class SmallCIFARNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            modified_conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
    
            modified_conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  
            
            modified_conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
        )
        self.flatten = nn.Flatten()
        self.softmax = nn.Softmax(dim = 1)
        self.classifier = nn.Sequential(
            modified_linear(128 * 4 * 4, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.quantize(k)

In [ ]:
import torch
from torch import nn
class mnist_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            modified_linear(28 * 28, 256),
            nn.ReLU(),
            modified_linear(256, 512),
            nn.ReLU(),
            modified_linear(512, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )
        
    def forward(self,x):
        x = self.flatten(x)
        x = self.classifier(x)
        return x
    
    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, modified_linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, modified_linear)):
                module.quantize(k)


In [ ]:
import torch.optim as optim
from torch import nn
from tqdm import tqdm
import torch
def train_model(model, train_loader,device,epochs=5,lr=0.01, mode="Training"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f}")

def evaluate(model, loader , device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

        
def train_and_eval(model, train_loader,test_loader,device,epochs=5,lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f}")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(100 * correct / total)
    return 100 * correct / total

In [ ]:
def save_model_npz(model, savepath):
    arrays = {}
    count = 0

    for name, module in model.named_modules():
        if not hasattr(module, "assignments"):
            continue
        if not hasattr(module, "hashtable"):
            continue
        if module.hashtable is None:
            continue

        count += 1
        prefix = name.replace(".", "_")

        arrays[f"{prefix}__shape"] = np.array(module.mask.shape)
        arrays[f"{prefix}__mask"] = module.mask.detach().cpu().numpy().astype(np.bool_)
        arrays[f"{prefix}__assignments"] = (
            module.assignments.detach().cpu().numpy().astype(np.uint8)
        )
        arrays[f"{prefix}__codebook"] = (
            module.hashtable.detach().cpu().numpy().astype(np.float32)
        )

        if module.bias is not None:
            arrays[f"{prefix}__bias"] = (
                module.bias.detach().cpu().numpy().astype(np.float32)
            )

    print(f"[save_model_npz] saved {count} layers")

    if count == 0:
        raise RuntimeError("NO COMPRESSED LAYERS FOUND")

    np.savez_compressed(savepath, **arrays)


def load_model_from_npz(model, npz_path, device="cpu"):
    raw = np.load(npz_path, allow_pickle=True)

    for name, module in model.named_modules():
        if not isinstance(module, (modified_linear, modified_conv2d)):
            continue

        prefix = name.replace(".", "_")

        key_mask = f"{prefix}__mask"
        if key_mask not in raw:
            continue   # layer not compressed

        # ---- LOAD DATA ----
        mask = torch.from_numpy(raw[f"{prefix}__mask"]).to(device)
        assignments = torch.from_numpy(raw[f"{prefix}__assignments"]).to(device)
        codebook = torch.from_numpy(raw[f"{prefix}__codebook"]).to(device)
        bias = (
            torch.from_numpy(raw[f"{prefix}__bias"]).to(device)
            if f"{prefix}__bias" in raw else None
        )

        # ---- COPY INTO MODULE ----
        module.mask.copy_(mask)
        module.assignments.copy_(assignments)

        module.hashtable = torch.nn.Parameter(
            codebook, requires_grad=False
        )

        if bias is not None and module.bias is not None:
            module.bias.data.copy_(bias)

        # ---- KILL DENSE WEIGHTS ----
        module.weight = None
        module.mode = "quantize"

        return model
    
def load_csr_from_npz(npz_path):
    raw = np.load(npz_path)
    csr_layers = {}

    # detect unique layer prefixes
    prefixes = set(k.split("__")[0] for k in raw.files)

    for prefix in prefixes:
        shape = tuple(raw[f"{prefix}__shape"])

        mask = raw[f"{prefix}__mask"]
        assignments = raw[f"{prefix}__assignments"]
        codebook = raw[f"{prefix}__codebook"]
        bias = raw[f"{prefix}__bias"] if f"{prefix}__bias" in raw else None

        # ---- Dense layer ----
        if len(shape) == 2:
            mask_2d = mask
            assign_2d = assignments
            out_shape = shape

        # ---- Conv2d layer ----
        elif len(shape) == 4:
            oc, ic, kh, kw = shape
            mask_2d = mask.reshape(oc, -1)
            assign_2d = assignments.reshape(oc, -1)
            out_shape = (oc, ic * kh * kw)

        else:
            continue

        rows, cols = np.nonzero(mask_2d)
        if rows.size == 0:
            continue

        csr = csr_matrix(
            (assign_2d[rows, cols], (rows, cols)),
            shape=out_shape
        )

        csr_layers[prefix] = {
            "csr": csr,
            "codebook": codebook,
            "bias": bias,
            "orig_shape": shape
        }

    return csr_layers


In [ ]:
import torch
import os
from pathlib import Path

def config_device():
    device = None
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(device)
    return device


In [ ]:

path = 'data' #path to the data folder
device = config_device()

train_loader,test_loader = MNIST_loader(path)
model = mnist_model()
model = model.to(device)

train_and_eval(model,train_loader,test_loader,device,epochs=1)
prune_model(model,0.95)
train_and_eval(model,train_loader,test_loader,device,epochs=1)

def count_mask_sparsity(model):
    total = 0
    zeros = 0
    for module in model.modules():
        if hasattr(module, "mask"):
            total += module.mask.numel()
            zeros += (module.mask == 0).sum().item()
    print(f"Masked sparsity: {100 * zeros / total:.2f}%")

count_mask_sparsity(model)
torch.save(model.state_dict(), "compressed_models/model.pth")
quantize_model(model,16)
train_and_eval(model,train_loader,test_loader,device,epochs=1)
npz_path = "compressed_models/compressed.npz"
save_model_npz(model,npz_path)

model2 = mnist_model()
model2 = load_model_from_npz(model,npz_path,device)
train_loader,test_loader = MNIST_loader(path)
model2 = model2.to(device)
print(evaluate(model2,test_loader,device))
print(model.classifier[0].__dict__.keys())
print(model.classifier[0].weight)

mps
Epoch [1/1] Loss: 0.4327
95.09
pruned_linear!!
pruned_linear!!
pruned_linear!!
pruned_linear!!
Epoch [1/1] Loss: 0.2198
95.89
Masked sparsity: 95.00%
quantized_linear!!
quantized_linear!!
quantized_linear!!
quantized_linear!!
Epoch [1/1] Loss: 0.1788
95.23
[save_model_npz] saved 4 layers
95.23
dict_keys(['training', '_parameters', '_buffers', '_non_persistent_buffers_set', '_backward_pre_hooks', '_backward_hooks', '_is_full_backward_hook', '_forward_hooks', '_forward_hooks_with_kwargs', '_forward_hooks_always_called', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_state_dict_hooks', '_state_dict_pre_hooks', '_load_state_dict_pre_hooks', '_load_state_dict_post_hooks', '_modules', 'in_features', 'out_features', 'mode'])
None
